# Bài toán A — TN-A1 (train lại JEFFNet trên PVF-10) + TN-A2 (chạy lên ảnh cắt của mình)

**Trước khi chạy:** Runtime → Change runtime type → GPU (A100 nếu có Pro; L4/T4 vẫn chạy được nhưng batch nhỏ hơn).
Thư mục `MyDrive/colab_A/` phải có: `PVF-10_112.zip`, `tam_cat.zip`, `JEFFNet.zip`.
Mọi kết quả chép về `MyDrive/colab_A/ket_qua/`. **Đứt phiên giữa chừng thì chạy lại ô 1 → 4:** ô 4 có `--resume`, nạp `last.pt` trên Drive và chạy tiếp từ epoch kế, không mất gì.

**Lượt v1 (A1_0, 14/09) ra F1 89,70 vì ba lệch so với Bảng II:** `train.py` công bố *chú thích mất* dòng `ColorJitter(brightness=0.5)` (ablation của chính tác giả không jitter: 89,62 — trùng số của mình); batch 32 thay vì 128; đứt ở epoch 68/100. Lượt v2 vá cả ba: bật lại jitter, tích luỹ gradient cho đủ 128, `--resume` + dừng sớm.

In [ ]:
# 1. Drive + giải nén
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess, zipfile, json, time, glob
GOC = '/content/drive/MyDrive/colab_A'
KQ  = f'{GOC}/ket_qua'; os.makedirs(KQ, exist_ok=True)
assert all(os.path.exists(f'{GOC}/{f}') for f in ['PVF-10_112.zip','tam_cat.zip','JEFFNet.zip']), 'thiếu zip trong MyDrive/colab_A'
for z in ['PVF-10_112.zip','tam_cat.zip','JEFFNet.zip']:
    with zipfile.ZipFile(f'{GOC}/{z}') as f: f.extractall('/content')
os.chdir('/content/JEFFNet')
print('PVF-10 train lớp:', sorted(os.listdir('/content/PVF-10/train')))
print('PVF-10 test  lớp:', sorted(os.listdir('/content/PVF-10/test')))
assert sorted(os.listdir('/content/PVF-10/train')) == sorted(os.listdir('/content/PVF-10/test')), 'tên lớp train/test phải giống nhau'
print('ảnh cắt:', sum(len(fs) for _,_,fs in os.walk('/content/tam_cat/ori')), 'ori |', sum(len(fs) for _,_,fs in os.walk('/content/tam_cat/mask')), 'mask')

In [ ]:
# 2. Thư viện + checkpoint StoP-JEPA ViT-B/16 (ImageNet-1k, 600 epoch) — lưu trên Drive để khỏi tải lại
!pip -q install timm wandb 2>/dev/null
STOP = f'{GOC}/vit-b16.pth.tar'
if not os.path.exists(STOP):
    !wget -q --show-progress -O "$STOP" "https://huggingface.co/amirbar1/StoP/resolve/main/vit-b16.pth.tar?download=true"
print('StoP ckpt: %.0f MB' % (os.path.getsize(STOP)/1e6))
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ — dừng lại, đổi runtime')

In [ ]:
# 3. Chọn batch theo GPU. Cấu hình gốc của tác giả: batch 128 trên A100-40GB.
ten = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 128 if vram >= 38 else (64 if vram >= 22 else 32)
ACCUM = 128 // BATCH  # tích luỹ gradient để batch HIỆU DỤNG luôn = 128 như tác giả (T4: 4×32, L4: 2×64)
print(f'{ten} ({vram:.0f} GB) → batch {BATCH} × accum {ACCUM} = {BATCH*ACCUM} hiệu dụng')
SEEDS = '0'          # thêm '0 1 2' nếu có thời gian; tác giả chạy 10 hạt
EPOCHS = 100         # đúng README (lịch cosine T_max=100 — đừng hạ, hạ là đổi lịch lr)
PATIENCE = 20        # dừng sớm: val_acc không tăng 20 epoch liền thì dừng. Best epoch của tác giả nằm ở 19–53/100
JITTER = 0.5         # ColorJitter(brightness) — dòng bị chú thích trong mã công bố; 0 = tắt để tái hiện lượt v1

In [ ]:
# 4. TN-A1 v2 — train lại theo cấu hình README (224 px, 10 epoch đóng băng + 90 mở, lr 3e-4 / 1e-5, CE có trọng số lớp)
# + 4 chỗ vá trong train.py (đánh dấu [vá TN-A1]): --color-jitter, --accum, --patience, --resume.
# ĐỨT PHIÊN: chạy lại ô 1, 2, 3 rồi ô này — --resume nạp RUN_<seed>/last.pt (ghi mỗi epoch, ~1,3 GB) và chạy tiếp.
# Lưu ý khi nối tiếp: thứ tự xáo trộn batch bắt đầu lại từ hạt, không giống hệt một lượt chạy liền — ghi vào bài nếu có nối.
os.environ['WANDB_MODE'] = 'disabled'
RUN = f'{KQ}/A1v2'   # v1 (A1_0: không jitter, batch 32, đứt ở 68/100) giữ nguyên làm đối chứng
t0 = time.time()
!python -u train.py --root-path /content --image-folder PVF-10 --dataset-layout split \
    --model-name deit_base --patch-size 16 --res 224 --pretrained "$STOP" --use-pretrained-eff \
    --seeds $SEEDS --epochs $EPOCHS --freeze-epochs 10 --batch-size $BATCH --accum $ACCUM \
    --color-jitter $JITTER --patience $PATIENCE --resume \
    --encoder-lr 1e-5 --head-lr 3e-4 --output-dir "$RUN" 2>&1 | grep --line-buffered -vE 'UserWarning|warnings.warn'
print('phiên này chạy %.1f phút' % ((time.time()-t0)/60))
hs = sorted(glob.glob(f'{RUN}_*/history.json'))
if hs:
    h = json.load(open(hs[0])); b = max(h, key=lambda r: r['val_acc'])
    print('đã xong %d/%d epoch; best val_acc %.4f ở epoch %d' % (len(h), EPOCHS, b['val_acc'], b['epoch']))
    print('→ nếu chưa thấy dòng [dừng sớm] và chưa đủ %d epoch: chạy lại ô 1→4 để nối tiếp' % EPOCHS)

In [ ]:
# 5. Đọc số A1: 10 lớp (mốc 93,21 ± 1,24 F1) và nhị phân gộp từ ma trận nhầm lẫn (mốc 97,53 F1) — đúng cách aggregate_binary.py của tác giả
import numpy as np
def nhi_phan(cm, h):
    cm = np.asarray(cm, float); k = cm.shape[0]; loi = [i for i in range(k) if i != h]
    TN = cm[h,h]; FP = cm[h,:].sum()-TN; FN = cm[:,h].sum()-TN; TP = cm[np.ix_(loi,loi)].sum()
    P = TP/(TP+FP) if TP+FP else float('nan'); R = TP/(TP+FN) if TP+FN else float('nan')
    return dict(acc=(TP+TN)/cm.sum(), P=P, R=R, F1=2*P*R/(P+R) if P+R else float('nan'))
tong = []
for d in sorted(glob.glob(f'{RUN}_*/test_metrics/metrics.json')):
    m = json.load(open(d)); ck = torch.load(os.path.dirname(os.path.dirname(d))+'/best_model.pt', map_location='cpu', weights_only=False)
    cls = ck['class_names']; h = [i for i,c in enumerate(cls) if 'healthy' in c.lower()][0]
    b = nhi_phan(m['confusion_matrix'], h)
    tong.append(dict(run=os.path.basename(os.path.dirname(os.path.dirname(d))), f1_10lop=m['metrics']['f1_macro'], acc_10lop=m['metrics']['accuracy'], f1_nhi_phan=b['F1'], acc_nhi_phan=b['acc']))
    print('%-40s 10 lớp F1 %.4f acc %.4f | nhị phân F1 %.4f acc %.4f' % (tong[-1]['run'], m['metrics']['f1_macro'], m['metrics']['accuracy'], b['F1'], b['acc']))
print('\nMỐC tác giả: 10 lớp F1 93,21 ± 1,24 / acc 94,33 · nhị phân F1 97,53 / acc 96,41')
h = json.load(open(sorted(glob.glob(f'{RUN}_*/history.json'))[0]))
json.dump(dict(phien_ban='v2', batch=BATCH, accum=ACCUM, batch_hieu_dung=BATCH*ACCUM, color_jitter=JITTER, patience=PATIENCE, gpu=ten, seeds=SEEDS,
               epochs_dat=EPOCHS, epochs_chay=len(h), best_epoch=max(h, key=lambda r: r['val_acc'])['epoch'], runs=tong),
          open(f'{KQ}/a1v2_tom_tat.json','w'), indent=1)

In [ ]:
# 6. TN-A2 — chạy checkpoint A1 lên 3.090 ảnh cắt / 614 tấm của mình, chấm nhị phân cấp tấm (best / any / major) + bootstrap
CKPT = sorted(glob.glob(f'{RUN}_*/best_model.pt'))[0]; print('checkpoint:', CKPT)
!python danh_gia_tam_cat.py --ckpt "$CKPT" --pretrained "$STOP" --tam-cat /content/tam_cat \
    --manifest /content/tam_cat/tam_cat_manifest.csv --out "$KQ/a2v2" --bien-the ori mask 2>&1 | grep -vE 'UserWarning|warnings.warn'
print('\nMỐC để đặt cạnh: pipeline mình trên cùng 614 tấm (ảnh lẻ, k=1): P 0,866 R 0,977 F1 0,918 [0,900–0,936]')

## Đọc kết quả
- `ket_qua/a1v2_tom_tat.json` — số A1 v2, có ghi batch/accum/jitter/patience/GPU thật đã dùng và số epoch thật chạy.
- `ket_qua/a2v2/a2_ori.json`, `a2_mask.json` — ba cách gộp lên tấm; `nhan_minh_sang_lop_ho` cho biết model gán lỗi của mình vào lớp nào của PVF-10.
- Tải hai file JSON đó về máy, đặt vào `ket_qua/` của dự án rồi chạy `py ma/cham_a2_ve_bang.py` (chưa viết) để điền Bảng A.